In [64]:
from pathlib import Path

%load_ext autoreload
%autoreload 2

import numpy as np
import pandas as pd
from wood_charts import load_theme
from wood_charts.charts import (
    area_chart,
    bar_chart,
    heatmap_chart,
    line_chart,
)

from synthetic_website_analytics.data import DatabaseConnector

theme = load_theme("notebook")

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [65]:
project_root = Path.cwd().parent
connector = DatabaseConnector.from_env(project_root / ".env")

In [66]:
query_path = project_root / "sql" / "performance" / "daily_performance.sql"
daily_performance = connector.execute_sql(query_path)

daily_performance["date_day"] = pd.to_datetime(daily_performance["date_day"])
daily_performance = daily_performance.loc[
    daily_performance["date_day"].between(
        pd.to_datetime("2026-01-13"),
        pd.to_datetime("2026-12-28"),
    )
].copy()

In [67]:
daily_performance = daily_performance.assign(
    session_conversion_rate=(
        daily_performance["purchase_session_count"] / daily_performance["session_count"]
    ),
    average_order_value=(
        daily_performance["total_order_value"] / daily_performance["order_count"]
    ),
    revenue_per_session=(
        daily_performance["total_order_value"] / daily_performance["session_count"]
    ),
)

daily_performance["sessions_7d_avg"] = (
    daily_performance["session_count"].rolling(7, min_periods=1).mean()
)

daily_performance["conversion_rate_7d_avg"] = (
    daily_performance["session_conversion_rate"].rolling(7, min_periods=1).mean()
)

daily_performance["revenue_7d_avg"] = (
    daily_performance["total_order_value"].rolling(7, min_periods=1).mean()
)

daily_performance["revenue_per_session_7d_avg"] = (
    daily_performance["revenue_per_session"].rolling(7, min_periods=1).mean()
)

In [68]:
figure = line_chart(
    daily_performance,
    x="date_day",
    y="session_count",
    theme=theme,
    title="Website Traffic Shows Strong Weekly Seasonality",
    subtitle="Daily sessions, January 2026-January 2027",
    x_axis_title="Date",
    y_axis_title="Sessions",
    source="Synthetic Website Data, marts.fct_website_daily_metrics",
)

figure.show()

In [69]:
daily_performance["day_of_week"] = daily_performance["date_day"].dt.day_name()

weekday_performance = daily_performance.groupby("day_of_week", as_index=False).agg(
    avg_sessions=("session_count", "mean"),
    avg_visitors=("visitor_count", "mean"),
)

day_order = [
    "Monday",
    "Tuesday",
    "Wednesday",
    "Thursday",
    "Friday",
    "Saturday",
    "Sunday",
]

daily_performance["day_type"] = np.where(
    daily_performance["date_day"].dt.dayofweek < 5,
    "Weekday",
    "Weekend",
)

day_type_performance = daily_performance.groupby("day_type", as_index=False).agg(
    avg_sessions=("session_count", "mean"),
    avg_visitors=("visitor_count", "mean"),
)

weekday_avg = day_type_performance.loc[
    day_type_performance["day_type"] == "Weekday",
    "avg_sessions",
].iloc[0]

weekend_avg = day_type_performance.loc[
    day_type_performance["day_type"] == "Weekend",
    "avg_sessions",
].iloc[0]

weekend_decline = 1 - (weekend_avg / weekday_avg)

weekday_performance["day_of_week"] = pd.Categorical(
    weekday_performance["day_of_week"],
    categories=day_order,
    ordered=True,
)

weekday_performance = weekday_performance.sort_values("day_of_week")

figure = bar_chart(
    weekday_performance,
    x="day_of_week",
    y="avg_sessions",
    theme=theme,
    title=f"Weekend Traffic Is {weekend_decline:.0%} Lower vs. Weekdays",
    subtitle="Weekend sessions compared with weekday average",
    x_axis_title="Day of Week",
    y_axis_title="Average Sessions",
    focus=["Saturday", "Sunday"],
    source="Synthetic Website Data, marts.fct_website_daily_metrics",
)
figure.show()

In [70]:
query_path = project_root / "sql" / "campaign" / "campaign_info.sql"
campaign_info = connector.execute_sql(query_path)

campaign_start = pd.to_datetime(campaign_info["campaign_start_date"].min())
campaign_end = pd.to_datetime(campaign_info["campaign_end_date"].max())
campaign_days = (campaign_end - campaign_start).days + 1

campaign_info

,campaign_id,campaign_start_date,campaign_end_date
0,display_awareness_spring,2026-03-15,2026-04-15
1,paid_search_spring_launch,2026-03-01,2026-03-31


In [71]:
figure = line_chart(
    daily_performance,
    x="date_day",
    y="sessions_7d_avg",
    theme=theme,
    title="Spring Traffic Peak Coincides With Campaign Activity",
    subtitle="7-day rolling average of daily sessions",
    x_axis_title="Date",
    y_axis_title="Sessions",
    event_bands=[
        {
            "start": campaign_info["campaign_start_date"].min(),
            "end": campaign_info["campaign_end_date"].max(),
            "label": "Spring campaign",
        }
    ],
    source="Synthetic Website Data, marts.fct_website_daily_metrics",
)

figure.show()

In [72]:
weekly_channel_sessions = (
    daily_performance.set_index("date_day")
    .resample("W-MON")
    .agg(
        paid_search_session_count=("paid_search_session_count", "sum"),
        display_session_count=("display_session_count", "sum"),
        unattributed_session_count=("unattributed_session_count", "sum"),
    )
    .reset_index()
)

figure = area_chart(
    weekly_channel_sessions,
    x="date_day",
    y=[
        "unattributed_session_count",
        "paid_search_session_count",
        "display_session_count",
    ],
    theme=theme,
    title="Paid Campaigns Drive the Spring Traffic Lift",
    subtitle="Weekly sessions by acquisition source",
    y_axis_title="Sessions",
    source="Synthetic Website Data, marts.fct_website_daily_metrics",
    event_bands=[
        {
            "start": campaign_start,
            "end": campaign_end,
            "label": "Spring campaign",
        }
    ],
    axis_break={"start": 0, "end": 2500},
)

figure.show()

In [73]:
query_path = project_root / "sql" / "navigation" / "page_navigation.sql"

page_navigation = connector.execute_sql(query_path)

page_order = [
    "home",
    "blog",
    "products",
    "product_detail",
    "cart",
    "checkout",
    "order_confirmation",
    "contact",
]

transition_heatmap = (
    page_navigation.pivot(
        index="to_page_name",
        columns="from_page_name",
        values="observed_transition_probability",
    )
    .reindex(
        index=page_order,
        columns=page_order,
        fill_value=0,
    )
    .fillna(0)
    .reset_index()
)

figure = heatmap_chart(
    transition_heatmap,
    x="to_page_name",
    y_columns=page_order,
    theme=theme,
    title="Visitors Follow a Clear Path Toward Checkout",
    subtitle="Observed probability of moving from each page to the next",
    source="Synthetic Website Data, marts.fct_website_page_navigation",
    x_axis_title="To Page",
    y_axis_title="From Page",
    colorbar_title="Transition Probability",
    mask_zero=True,
    annotate=True,
    value_format=".0%",
    zmin=0,
    zmax=1,
    format_labels=True,
)

figure.show()

In [74]:
total_sessions = daily_performance["session_count"].sum()

conversion_funnel = pd.DataFrame(
    {
        "stage": [
            "Sessions",
            "Product Views",
            "Add to Cart",
            "Begin Checkout",
            "Purchases",
        ],
        "conversion_rate": [
            100,
            daily_performance["product_view_session_count"].sum()
            / total_sessions
            * 100,
            daily_performance["add_to_cart_session_count"].sum() / total_sessions * 100,
            daily_performance["begin_checkout_session_count"].sum()
            / total_sessions
            * 100,
            daily_performance["purchase_session_count"].sum() / total_sessions * 100,
        ],
    }
)

figure = bar_chart(
    conversion_funnel,
    x="conversion_rate",
    y="stage",
    orientation="horizontal",
    theme=theme,
    title="Website Conversion Funnel",
    subtitle="Share of all sessions that reach each step in the purchase funnel",
    x_axis_title="Sessions Reaching Stage",
    y_axis_title="Funnel Stage",
    show_values=True,
    source="Synthetic Website Data, marts.fct_website_daily_metrics",
)

figure.update_yaxes(autorange="reversed")
figure.update_xaxes(ticksuffix="%")
figure.update_traces(texttemplate="%{text:.1f}%")
figure.show()

In [75]:
figure = line_chart(
    daily_performance,
    x="date_day",
    y="conversion_rate_7d_avg",
    theme=theme,
    title="Conversion Remains Stable Despite the Spring Traffic Surge",
    subtitle="7-day rolling average of session conversion rate",
    x_axis_title="Date",
    y_axis_title="Session Conversion Rate",
    event_bands=[
        {
            "start": campaign_start,
            "end": campaign_end,
            "label": "Spring campaign",
        }
    ],
    source="Synthetic Website Data, marts.fct_website_daily_metrics",
)

figure.update_yaxes(tickformat=".0%")
figure.update_traces(hovertemplate="%{x|%b %d, %Y}<br>%{y:.1%}<extra></extra>")
figure.show()

In [76]:
figure = line_chart(
    daily_performance,
    x="date_day",
    y="revenue_7d_avg",
    theme=theme,
    title="Spring Campaigns Produce a Sustained Revenue Lift",
    subtitle="7-day rolling average of daily revenue",
    x_axis_title="Date",
    y_axis_title="Revenue",
    event_bands=[
        {
            "start": campaign_start,
            "end": campaign_end,
            "label": "Spring campaign",
        }
    ],
    source="Synthetic Website Data, marts.fct_website_daily_metrics",
)

figure.update_yaxes(tickprefix="$", tickformat=",.0f")
figure.update_traces(hovertemplate="%{x|%b %d, %Y}<br>$%{y:,.0f}<extra></extra>")
figure.show()

In [77]:
campaign_performance = daily_performance.loc[
    daily_performance["date_day"].between(campaign_start, campaign_end)
].copy()
campaign_revenue_per_session = (
    campaign_performance["total_order_value"].sum()
    / campaign_performance["session_count"].sum()
)

figure = line_chart(
    daily_performance,
    x="date_day",
    y="revenue_per_session_7d_avg",
    theme=theme,
    title=(
        f"Campaign Traffic Generates ${campaign_revenue_per_session:,.2f} "
        "Revenue per Session"
    ),
    subtitle="7-day rolling average of revenue generated by each session",
    x_axis_title="Date",
    y_axis_title="Revenue per Session",
    event_bands=[
        {
            "start": campaign_start,
            "end": campaign_end,
            "label": "Spring campaign",
        }
    ],
    source="Synthetic Website Data, marts.fct_website_daily_metrics",
)

figure.update_yaxes(tickprefix="$", tickformat=",.0f")
figure.update_traces(hovertemplate="%{x|%b %d, %Y}<br>$%{y:,.2f}<extra></extra>")
figure.show()

In [79]:
pre_campaign_start = campaign_start - pd.Timedelta(days=campaign_days)
post_campaign_end = campaign_end + pd.Timedelta(days=campaign_days)

campaign_periods = {
    "Pre-campaign": (
        pre_campaign_start,
        campaign_start - pd.Timedelta(days=1),
    ),
    "Campaign": (campaign_start, campaign_end),
    "Post-campaign": (
        campaign_end + pd.Timedelta(days=1),
        post_campaign_end,
    ),
}

campaign_lift_summary = pd.DataFrame(
    [
        {
            "period": period,
            "sessions": period_data["session_count"].sum(),
            "conversion_rate": (
                period_data["purchase_session_count"].sum()
                / period_data["session_count"].sum()
            ),
            "average_order_value": (
                period_data["total_order_value"].sum()
                / period_data["order_count"].sum()
            ),
            "revenue_per_session": (
                period_data["total_order_value"].sum()
                / period_data["session_count"].sum()
            ),
            "revenue_per_day": (
                period_data["total_order_value"].sum() / len(period_data)
            ),
        }
        for period, (start_date, end_date) in campaign_periods.items()
        for period_data in [
            daily_performance.loc[
                daily_performance["date_day"].between(start_date, end_date)
            ]
        ]
    ]
)

campaign_lift_table = campaign_lift_summary.set_index("period").T.rename(
    index={
        "sessions": "Sessions",
        "conversion_rate": "Conversion Rate",
        "average_order_value": "AOV",
        "revenue_per_session": "Revenue per Session",
        "revenue_per_day": "Revenue per Day",
    }
)

campaign_lift_display = campaign_lift_table.astype(object)

campaign_lift_display.loc["Sessions"] = campaign_lift_display.loc["Sessions"].map(
    lambda value: f"{value:,.0f}"
)
campaign_lift_display.loc["Conversion Rate"] = campaign_lift_display.loc[
    "Conversion Rate"
].map(lambda value: f"{value:.1%}")
for metric in ["AOV", "Revenue per Session", "Revenue per Day"]:
    campaign_lift_display.loc[metric] = campaign_lift_display.loc[metric].map(
        lambda value: f"${value:,.2f}"
    )


pre_campaign = campaign_lift_summary.loc[
    campaign_lift_summary["period"] == "Pre-campaign"
].iloc[0]
campaign = campaign_lift_summary.loc[
    campaign_lift_summary["period"] == "Campaign"
].iloc[0]

revenue_change = campaign["revenue_per_day"] / pre_campaign["revenue_per_day"] - 1
session_change = campaign["sessions"] / pre_campaign["sessions"] - 1
conversion_change = campaign["conversion_rate"] - pre_campaign["conversion_rate"]
aov_change = campaign["average_order_value"] / pre_campaign["average_order_value"] - 1

campaign_lift_narrative = (
    f"Campaign-period revenue per day increased {revenue_change:.0%}, driven "
    f"primarily by a {session_change:.0%} increase in sessions. Conversion "
    f"changed {conversion_change:+.1%} and AOV changed {aov_change:+.0%}."
)


display(campaign_lift_display)

Campaign-period revenue per day increased 18%, driven primarily by a 18% increase in sessions. Conversion changed -0.2% and AOV changed +1%.


period,Pre-campaign,Campaign,Post-campaign
Sessions,"18,300","21,515","17,993"
Conversion Rate,20.8%,20.6%,20.5%
AOV,$236.85,$239.28,$236.16
Revenue per Session,$49.22,$49.27,$48.43
Revenue per Day,"$19,581.67","$23,043.53","$18,943.96"
